In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_utils import load_mnist_vae_data, get_device, seed_everything
from src.models import ConvVAE
from src.evaluate import reconstruct_images
from src.latent_analysis import encode_dataset

In [ ]:
seed_everything(42)
device = get_device()
device

In [ ]:
best_config = {
    "latent_dim": 16,
    "beta": 1.0,
    "epochs": 20,
    "lr": 1e-3,
    "hidden_dim": 256,
    "base_channels": 64,
    "batch_size": 512,
    "warmup_epochs": None,
    "recon_loss_type": "bce",
}
best_config

In [ ]:
train_loader, val_loader, test_loader = load_mnist_vae_data(
    data_dir="../data",
    batch_size=best_config["batch_size"],
    val_ratio=0.1,
    num_workers=0,
    random_state=42,
)

In [ ]:
vae_ckpt_path = "../results/logs/h1_conv_vae_best_latent16.pt"

vae16 = ConvVAE(
    input_channels=1,
    latent_dim=16,
    hidden_dim=best_config["hidden_dim"],
    base_channels=best_config["base_channels"],
).to(device)

state_dict = torch.load(vae_ckpt_path, map_location=device)
vae16.load_state_dict(state_dict)
vae16.eval()

print(f"Loaded VAE checkpoint from: {vae_ckpt_path}")

In [ ]:
def loader_to_numpy(data_loader):
    xs, ys = [], []
    for x, y in data_loader:
        xs.append(x.view(x.size(0), -1).numpy())
        ys.append(y.numpy())
    X = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return X, y

X_train, y_train = loader_to_numpy(train_loader)
X_test, y_test = loader_to_numpy(test_loader)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

In [ ]:
def fit_pca_and_reconstruct(X_train, X_test, n_components):
    pca = PCA(n_components=n_components, random_state=42)
    Z_train = pca.fit_transform(X_train)
    Z_test = pca.transform(X_test)
    X_test_recon = pca.inverse_transform(Z_test)
    return pca, Z_train, Z_test, X_test_recon

def mse_per_sample_np(x_true, x_pred):
    return np.mean((x_true - x_pred) ** 2, axis=1)

def bce_per_sample_np(x_true, x_pred, eps=1e-8):
    x_pred = np.asarray(x_pred, dtype=np.float64)
    x_true = np.asarray(x_true, dtype=np.float64)

    # PCA 重构可能超出 [0, 1]，先截断
    x_pred = np.clip(x_pred, eps, 1.0 - eps)
    x_true = np.clip(x_true, 0.0, 1.0)

    return -np.sum(
        x_true * np.log(x_pred) + (1.0 - x_true) * np.log(1.0 - x_pred),
        axis=1
    )

In [ ]:
pca16, Z_train_16, Z_test_16, X_test_recon_pca16 = fit_pca_and_reconstruct(
    X_train, X_test, n_components=16
)

pca2, Z_train_2, Z_test_2, X_test_recon_pca2 = fit_pca_and_reconstruct(
    X_train, X_test, n_components=2
)

print("PCA-16 explained variance ratio sum:", pca16.explained_variance_ratio_.sum())
print("PCA-2  explained variance ratio sum:", pca2.explained_variance_ratio_.sum())

In [ ]:
@torch.no_grad()
def reconstruct_loader_full(model, data_loader, device):
    model.eval()
    x_true_list = []
    x_recon_list = []
    y_list = []

    for x, y in data_loader:
        recon_x, mu, logvar = model(x.to(device))
        x_true_list.append(x.cpu().view(x.size(0), -1).numpy())
        x_recon_list.append(recon_x.cpu().view(x.size(0), -1).numpy())
        y_list.append(y.numpy())

    X_true = np.concatenate(x_true_list, axis=0)
    X_recon = np.concatenate(x_recon_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    return X_true, X_recon, y_all

X_test_true_vae, X_test_recon_vae16, y_test_vae = reconstruct_loader_full(
    vae16, test_loader, device
)

print(X_test_true_vae.shape, X_test_recon_vae16.shape)

In [ ]:
results = []

for name, X_recon in [
    ("PCA-2", X_test_recon_pca2),
    ("PCA-16", X_test_recon_pca16),
    ("VAE-16", X_test_recon_vae16),
]:
    mse_vals = mse_per_sample_np(X_test, X_recon)
    bce_vals = bce_per_sample_np(X_test, X_recon)

    results.append({
        "method": name,
        "mean_mse": float(np.mean(mse_vals)),
        "std_mse": float(np.std(mse_vals)),
        "mean_bce": float(np.mean(bce_vals)),
        "std_bce": float(np.std(bce_vals)),
    })

results_df = pd.DataFrame(results).sort_values("mean_mse")
results_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results_df["method"], results_df["mean_mse"])
plt.ylabel("Mean reconstruction MSE")
plt.title("Reconstruction MSE: PCA vs VAE")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results_df["method"], results_df["mean_bce"])
plt.ylabel("Mean reconstruction BCE")
plt.title("Reconstruction BCE: PCA vs VAE")
plt.tight_layout()
plt.show()

In [ ]:
def show_method_comparison(
    X_true,
    recon_dict,
    y,
    indices=None,
    n=8,
    save_path=None,
):
    if indices is None:
        rng = np.random.default_rng(42)
        indices = rng.choice(len(X_true), size=n, replace=False)
    else:
        indices = np.array(indices[:n])

    n = len(indices)
    n_rows = 1 + len(recon_dict)

    fig, axes = plt.subplots(n_rows, n, figsize=(1.8 * n, 1.8 * n_rows))

    if n_rows == 1:
        axes = np.array([axes])

    method_names = ["Original"] + list(recon_dict.keys())

    for j, idx in enumerate(indices):
        axes[0, j].imshow(X_true[idx].reshape(28, 28), cmap="gray")
        axes[0, j].axis("off")
        axes[0, j].set_title(f"y={y[idx]}")

    for i, (method, X_recon) in enumerate(recon_dict.items(), start=1):
        for j, idx in enumerate(indices):
            axes[i, j].imshow(X_recon[idx].reshape(28, 28), cmap="gray")
            axes[i, j].axis("off")

    for i, method_name in enumerate(method_names):
        axes[i, 0].set_ylabel(method_name, rotation=90, size=11)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
os.makedirs("../results/figures/h2_vae_vs_pca", exist_ok=True)

show_method_comparison(
    X_true=X_test,
    recon_dict={
        "PCA-2": X_test_recon_pca2,
        "PCA-16": X_test_recon_pca16,
        "VAE-16": X_test_recon_vae16,
    },
    y=y_test,
    n=10,
    save_path="../results/figures/h2_vae_vs_pca/h2_reconstruction_comparison.png",
)

In [ ]:
plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    Z_test_2[:, 0],
    Z_test_2[:, 1],
    c=y_test,
    s=8,
    alpha=0.7,
)
plt.colorbar(scatter)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA-2 projection on MNIST test set")
plt.tight_layout()
plt.savefig("../results/figures/h2_vae_vs_pca/h2_pca2_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
vae2_ckpt_path = "../results/logs/h1_conv_vae_best_latent2.pt"
use_vae2 = os.path.exists(vae2_ckpt_path)
print("Found VAE-2 checkpoint:", use_vae2)

In [ ]:
if use_vae2:
    vae2 = ConvVAE(
        input_channels=1,
        latent_dim=2,
        hidden_dim=best_config["hidden_dim"],
        base_channels=best_config["base_channels"],
    ).to(device)

    vae2.load_state_dict(torch.load(vae2_ckpt_path, map_location=device))
    vae2.eval()

    encoded_test_vae2 = encode_dataset(vae2, test_loader, device=device, use_mu=True)
    z_test_vae2 = encoded_test_vae2["mu"]
    y_test_vae2 = encoded_test_vae2["y"]

    plt.figure(figsize=(7, 6))
    scatter = plt.scatter(
        z_test_vae2[:, 0],
        z_test_vae2[:, 1],
        c=y_test_vae2,
        s=8,
        alpha=0.7,
    )
    plt.colorbar(scatter)
    plt.xlabel("z1")
    plt.ylabel("z2")
    plt.title("VAE-2 latent means on MNIST test set")
    plt.tight_layout()
    plt.savefig("../results/figures/h2_vae_vs_pca/h2_vae2_scatter.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("VAE-2 checkpoint not found. Skip VAE-2 visualization for now.")

In [ ]:
os.makedirs("../results/tables", exist_ok=True)
results_df.to_csv("../results/tables/h2_vae_vs_pca_reconstruction_metrics.csv", index=False)
results_df

In [ ]:
summary_lines = [
    "H2 结论草稿：",
    "1. PCA 与 VAE 都能在低维表示下实现图像重构，但 VAE-16 通常能给出更自然的数字轮廓。",
    "2. PCA-2 更接近线性投影几何，类别结构有一定可分性，但重构能力有限。",
    "3. VAE-2 / VAE-16 学到的是非线性潜在表示，既保留了局部连续性，也更适合后续生成与聚类分析。",
    "4. 因而，相较于 PCA，卷积 VAE 在 MNIST 上体现出更强的非线性表示能力与可生成性。",
]
print("\n".join(summary_lines))